

# Sensitivity Analysis: Chairs, Desks and Tables
### OPIM 5641 - Business Decision Modeling · Module 3
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5641-notebooks/blob/main/6_Pyomo_LP/5_SensitivityAnalysis_Furniture.ipynb)

*Run me top to bottom - **Runtime → Run all**. Pyomo and a solver install in the first cell.*

*Veerman Furniture - Section 9.4 (Powell).*

This notebook takes the sensitivity analysis from `4_BlendingModels_Furniture` and gives it room to breathe. Two differences on purpose:

1. We use the **plain product-mix model** - the one you brute-forced back in M2.1 - with no 25% balance constraints in the way. Same problem, same \$8,400, so you can see the sensitivity ideas without a second thing going on.
2. The constraints get **real names** instead of `Constraint1`, `Constraint2`, `Constraint3`, which makes every table below readable.

**The question this notebook answers** is the one you actually get asked in a meeting:

> *"Fabrication can run overtime at \$6 an hour. Should we take it?"*

The optimal plan can't answer that. Sensitivity analysis can.

🔴
<!-- 🎙 DAVE TALKING POINTS (invisible when rendered - double-click to read):
- OPEN: "you have solved this problem three ways and gotten $8,400 every time. Today the answer stops being the point."
- The meeting question: fabrication offers overtime at $6/hour. Yes or no? The optimal plan alone cannot answer it.
- Say why this notebook exists separately from the blending one: plain model, no 25% balance constraints, so sensitivity is the only new idea on screen.
- And the constraints get real names - that is the "if we were cuter with our names" note from the blending notebook, finally done.
-->

## Setup

In [ ]:
# import modules

%matplotlib inline
from pylab import *

import shutil
import sys
import os.path

if not shutil.which("pyomo"):
    !pip install -q pyomo
    assert(shutil.which("pyomo"))

if not (shutil.which("cbc") or os.path.isfile("cbc")):
    if "google.colab" in sys.modules:
        !apt-get install -y -qq coinor-cbc
    else:
        try:
            !conda install -c conda-forge coincbc 
        except:
            pass

assert(shutil.which("cbc") or os.path.isfile("cbc"))

from pyomo.environ import *

## The model

$$Max\ Z = 15C + 24D + 18T$$

subject to

* $4C + 6D + 2T \le 1{,}850$  (fabrication hours)
* $3C + 5D + 7T \le 2{,}400$  (assembly hours)
* $3C + 2D + 4T \le 1{,}500$  (shipping hours)
* $C \le 360$, $D \le 300$, $T \le 100$  (demand ceilings)
* $C, D, T \ge 0$

In [ ]:
# declare the model
model = ConcreteModel()

# declare decision variables
model.c = Var(domain=NonNegativeReals)
model.d = Var(domain=NonNegativeReals)
model.t = Var(domain=NonNegativeReals)

# declare objective
model.profit = Objective(
                    expr = 15*model.c + 24*model.d + 18*model.t,
                    sense = maximize)

# declare constraints - giving them real names pays off in the tables below
model.fabrication = Constraint(expr = 4*model.c + 6*model.d + 2*model.t <= 1850)
model.assembly    = Constraint(expr = 3*model.c + 5*model.d + 7*model.t <= 2400)
model.shipping    = Constraint(expr = 3*model.c + 2*model.d + 4*model.t <= 1500)
model.cDemand     = Constraint(expr = model.c <= 360)
model.dDemand     = Constraint(expr = model.d <= 300)
model.tDemand     = Constraint(expr = model.t <= 100)

# solve it
SolverFactory('cbc', executable='/usr/bin/cbc').solve(model)

# show the results
print('Profit = $', model.profit())
print('Chairs = ', model.c())
print('Desks  = ', model.d())
print('Tables = ', model.t())

**\$8,400, at 0 chairs, 275 desks and 100 tables.**

That's the same answer brute force ground out of 10,974,761 combinations in M2.1 - and the same answer you'd get graphically or with a Simplex tableau. Now let's ask what the plan can't tell us.

## Slack: which constraints are actually binding?

Pyomo will hand you the slack on any constraint with `.uslack()` - how much room is left under an upper limit. A constraint with **zero slack is binding**: the plan used every last hour of it.

**Only binding constraints are holding your profit down.** Relaxing a constraint you weren't using does nothing at all.

In [ ]:
str = "{0:>12.1f} {1:>12.1f} {2:>12.1f}"

myConstraints = [model.fabrication, model.assembly, model.shipping,
                 model.cDemand, model.dDemand, model.tDemand]

print("%-14s %12s %12s %12s   %s" % ('constraint','value','lslack','uslack','status'))
for c in myConstraints:
  status = 'BINDING' if c.uslack() < 0.0001 else ''
  print("%-14s" % c.name, str.format(c(), c.lslack(), c.uslack()), '  ', status)

Read that table and the whole picture changes:

* **Fabrication is binding** - all 1,850 hours used.
* **Assembly has 325 hours spare. Shipping has 550.** Nobody needs to buy you more shipping capacity; you aren't using what you have.
* **The table demand ceiling is binding too** - you're building all 100 tables you're allowed to sell.

**Remember:** this is exactly what you read off the bottom of the Simplex tableau in M2.3. A slack variable that went to zero meant a binding constraint. Same idea - Pyomo just does the arithmetic.

🔴
<!-- 🎙 DAVE TALKING POINTS (invisible when rendered - double-click to read):
- This table is the payoff - read it out loud. Fabrication used all 1,850. Assembly 325 spare. Shipping 550 spare.
- The business line: "nobody needs to buy you more shipping capacity - you are not using what you have."
- Note the table demand ceiling is ALSO binding - you build all 100 tables you are allowed to sell.
- Tie hard to M2.3: a slack variable that went to zero in the tableau meant a binding constraint. Identical idea, Pyomo does the arithmetic.
- lslack vs uslack: our constraints are all <=, so uslack is the one that matters. lslack is for >= limits.
- CLOSE: "fabrication is the bottleneck. So what is one more fabrication hour actually WORTH?"
-->

# For Loop for Shadow Prices

The **shadow price** of a constraint is what one more unit of it is worth to you.

You could ask a solver for it. **We're going to just measure it instead** - re-solve the whole model at a range of fabrication hours, and watch what profit does. Same for-loop habit you've had since M1.2, and there's nothing hidden in it.

In [ ]:
myHours = np.arange(1500, 2400, 10)
print(myHours)

In [ ]:
# store the results
import pandas as pd
myResults = pd.DataFrame()

for a in myHours:
  # declare the model
  model = ConcreteModel()

  # declare decision variables
  model.c = Var(domain=NonNegativeReals)
  model.d = Var(domain=NonNegativeReals)
  model.t = Var(domain=NonNegativeReals)

  # declare objective
  model.profit = Objective(
                      expr = 15*model.c + 24*model.d + 18*model.t,
                      sense = maximize)

  # declare constraints - 'a' is the fabrication hours we are testing
  model.fabrication = Constraint(expr = 4*model.c + 6*model.d + 2*model.t <= a)
  model.assembly    = Constraint(expr = 3*model.c + 5*model.d + 7*model.t <= 2400)
  model.shipping    = Constraint(expr = 3*model.c + 2*model.d + 4*model.t <= 1500)
  model.cDemand     = Constraint(expr = model.c <= 360)
  model.dDemand     = Constraint(expr = model.d <= 300)
  model.tDemand     = Constraint(expr = model.t <= 100)

  # solve it (quietly this time - 90 solves is a lot of output)
  SolverFactory('cbc', executable='/usr/bin/cbc').solve(model)

  # results
  myX = pd.DataFrame([a, model.profit(), model.c(), model.d(), model.t()])
  myX = myX.T

  # store the profit
  myResults = pd.concat([myResults, myX])

# Change the Column names
myResults = myResults.rename( {0:"Fabrication Hours", 1:"Profit", 2:"Chairs", 3:"Desks", 4:"Tables"}, axis='columns')
myResults.reset_index(drop=True, inplace=True)
myResults.head()

**Caution:** older versions of this code used `myResults.append(myX)`. **`DataFrame.append` was removed in pandas 2.0**, so it now throws `AttributeError` in Colab. `pd.concat([myResults, myX])` is the replacement and does the same job. If you meet `.append()` in an old notebook of mine, that's the fix.

## Profit - and the shadow price

In [ ]:
# add a column for marginal increase
# if you wanted it by ONE HOUR
myResults['ProfitDiff'] = myResults['Profit'].diff()/myResults['Fabrication Hours'].diff()

# show the table
myResults.head(15)

In [ ]:
# make a nice plot
tmp = myResults.drop(['Chairs','Desks','Tables','ProfitDiff'], axis=1)
tmp.plot(x='Fabrication Hours', figsize=(7,4), title='Profit vs. fabrication hours')

In [ ]:
# and the marginal value of one more hour
tmp = myResults.drop(['Profit','Chairs','Desks','Tables'], axis=1)
tmp.plot(x='Fabrication Hours', figsize=(7,4), title='What is one more hour worth?')

**That `ProfitDiff` column IS the shadow price** - measured, not asserted. Around our current 1,850 hours it reads **\$4.00 an hour**.

So the meeting question answers itself: **overtime at \$6/hour is a bad deal.** You'd be paying \$6 for something worth \$4 and losing \$2 every hour. At \$3/hour you'd take all of it.

🔴
<!-- 🎙 DAVE TALKING POINTS (invisible when rendered - double-click to read):
- The ProfitDiff column IS the shadow price - measured, not asserted. That is the whole move.
- My line: "you could ask a solver for it. We are going to just measure it instead - there is nothing hidden in a for loop."
- Read the number at 1,850: $4.00 per hour.
- ANSWER THE MEETING QUESTION: overtime at $6/hour is a BAD deal - paying $6 for something worth $4. At $3/hour you take all of it.
- Mention the pandas fix: .append() was removed in pandas 2.0, pd.concat is the replacement. They will hit this in old notebooks.
- CLOSE: "but look at the second plot - that line is not flat. How far can you trust the $4?"
-->

## The part that catches people: a shadow price is only good over a *range*

Look at the second plot again. **The value of an extra hour is not a constant - it's a staircase, and it steps down.**

| Fabrication hours | Worth per hour | Why |
|---|---|---|
| up to **2,000** | **\$4.00** | every extra hour goes into more desks |
| beyond 2,000 | \$3.75 | desks are maxed out, so you're pushed into chairs |
| beyond ~2,270 | less again | the next constraint starts to bite |

So the \$4.00 is good from 1,850 up to 2,000 - a **range of 150 hours**. Walk into a negotiation and ask for 500 hours of overtime on the strength of a \$4.00 shadow price and you'll overpromise, because the last 350 of them aren't worth \$4.

In [ ]:
# what happens right around 2,000 hours?
myResults[(myResults['Fabrication Hours'] >= 1960) & (myResults['Fabrication Hours'] <= 2060)]

**Remember:** the breakpoint isn't arbitrary. $6 \times 300$ desks $+\ 2 \times 100$ tables $= 2{,}000$ fabrication hours - so **2,000 is exactly where desks hit their demand ceiling of 300.** Up to there, an extra hour buys another slice of a \$24 desk. After it, desks are maxed and the hour has to go into chairs, which pay less per hour of fabrication.

**The shadow price changes exactly when the optimal product mix has to change.**

🔴
<!-- 🎙 DAVE TALKING POINTS (invisible when rendered - double-click to read):
- Slow down here - this is the most important idea in the notebook.
- The $4.00 is valid from 1,850 to exactly 2,000 hours. A 150-hour range.
- WHY 2,000? 6 x 300 desks + 2 x 100 tables = 2,000. Desks hit their demand ceiling. Show the table - desks pin at 300 and chairs start appearing.
- Punchline: the shadow price changes exactly when the optimal product MIX has to change.
- Business warning: ask for 500 hours of overtime on a $4.00 shadow price and you overpromise - the last 350 are not worth $4.
- CLOSE: "this is how you stop being the person who quotes one number in a meeting and gets it wrong."
-->

## Product Mix

In [ ]:
# make a nice plot
tmp = myResults.drop(['Profit','ProfitDiff'], axis=1)
tmp.plot(x='Fabrication Hours', figsize=(7,4.5), title='What you build changes with capacity')

Tables sit flat at 100 the whole way - they're capped by demand from the start. Desks climb until they hit 300 and stop dead. And **chairs are zero until 2,000 hours, then start being worth building.**

That's a satisfying answer to the objection you raised way back in M2.1, when the optimizer said build **no chairs at all**. It isn't that chairs are bad. It's that at 1,850 fabrication hours you have something better to do with the capacity - and the moment you have enough hours, chairs show up on their own.

# On your own

1. **Assembly and shipping aren't binding.** How many hours would you have to *take away* before one of them starts to bite? Sweep downward and find out.
2. The **table demand ceiling** is binding. What's one more table's worth of demand worth to you? Re-run the sweep over `tDemand` instead of fabrication and read the `ProfitDiff`.
3. **Change a price, not a capacity.** If desks fell from \$24 to \$20, does the plan change? How far can that number drop before the mix moves?
4. Run the same sweep on the **blending version** in `4_BlendingModels_Furniture` - with the 25% balance constraints in play, does fabrication still have the same shadow price? Why not?

## Bottom line

The optimal plan is one number. **Sensitivity analysis is the conversation around it.**

- A **binding** constraint has zero slack, and it's the only kind holding your profit down.
- Its **shadow price** is what one more unit is worth - and you can just *measure* it with a for loop instead of trusting a black box.
- That shadow price is only good over a **range**, and the range ends when the optimal mix has to change.
- A **non-binding** constraint is worth nothing at the margin. More of it changes nothing.

You've now met slack and binding constraints three times: by hand in the Simplex tableau (M2.3), in Pyomo's `uslack()`, and in a plot. Same idea every time.